In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import json
import os

# 1. Tentukan lokasi file
ROOT_DIR = "/content/drive/MyDrive/SFT/ready_to_train"
FILE_JSON_LAMA = os.path.join(ROOT_DIR, "final_pilgan_matematika.json")
FILE_JSON_BARU = os.path.join(ROOT_DIR, "sudah_replace/final_pilgan_matematika_re.json")

# Tentukan folder tempat kamu menyimpan 3 file system_prompt.md tersebut
# (Ubah jika lokasinya berbeda)
DIR_SYSTEM_PROMPT = os.path.join(ROOT_DIR, "system_prompts_train")

def baca_md(path_file):
    try:
        with open(path_file, "r", encoding="utf-8") as f:
            return f.read().strip()
    except FileNotFoundError:
        print(f"Peringatan: File {path_file} tidak ditemukan!")
        return ""

# 2. Buka data mentah
with open(FILE_JSON_LAMA, "r", encoding="utf-8") as f:
    data_mentah = json.load(f)

print(f"Ditemukan {len(data_mentah)} data.")

# 3. Looping untuk me-replace masing-masing data
for chat in data_mentah:
    meta = chat.get("metadata", {})
    if not meta:
        continue

    # Ambil task_type (materi, pilgan, atau essay) dan path sumbernya
    task_type = meta.get("tipe_konteks", "materi")
    path_sumber = meta.get("path_file_sumber", "")
    nama_file_saja = os.path.basename(path_sumber)

    # ---------------------------------------------------------
    # BAGIAN 1: MENGAMBIL TEKS ASLI DARI PATH FILE SUMBER (TETAP)
    # ---------------------------------------------------------
    teks_konteks = ""
    if os.path.exists(path_sumber):
        with open(path_sumber, "r", encoding="utf-8") as f_sumber:
            data_json = json.load(f_sumber)
            isi_assistant = data_json.get("assistant", {})

            if task_type == "materi":
                teks_konteks = isi_assistant.get("konten_markdown", "")
            elif task_type == "pilgan":
                for idx, soal in enumerate(isi_assistant):
                    teks_konteks += f"\nSoal {idx+1}:\nStimulus: {soal.get('stimulus')}\n"
                    teks_konteks += f"Pertanyaan: {soal.get('question')}\n"
                    teks_konteks += f"Opsi: {json.dumps(soal.get('options', {}), indent=2)}\n"
                    teks_konteks += f"Kunci Jawaban: {soal.get('answer')}\n"
                    teks_konteks += f"Penjelasan: {soal.get('explanation')}\n"
            elif task_type == "essay":
                for idx, soal in enumerate(isi_assistant):
                    teks_konteks += f"\nSoal {idx+1}:\nStimulus: {soal.get('stimulus')}\n"
                    teks_konteks += f"Pertanyaan: {soal.get('question')}\n"
                    teks_konteks += f"Penjelasan: {soal.get('explanation')}\n"
    else:
        teks_konteks = "[FILE MATERI TIDAK DITEMUKAN DI DRIVE]"

    # ---------------------------------------------------------
    # BAGIAN 2: MENGGANTI SYSTEM PROMPT
    # ---------------------------------------------------------
    # Otomatis menyesuaikan nama file (materi_system_prompt.md, dll)
    nama_file_sys = f"{task_type}_system_prompt.md"
    path_sys_prompt = os.path.join(DIR_SYSTEM_PROMPT, nama_file_sys)

    # Baca isi file md tersebut secara utuh
    teks_system_baru = baca_md(path_sys_prompt)

    # Timpa teks "ips_sys_materi" dengan teks asli dari file .md
    if len(chat["messages"]) > 0 and chat["messages"][0]["role"] == "system":
        chat["messages"][0]["content"] = teks_system_baru

    # ---------------------------------------------------------
    # BAGIAN 3: MENGGANTI NAMA FILE DI USER PROMPT & TAMBAH ENTER
    # ---------------------------------------------------------
    if len(chat["messages"]) > 1 and chat["messages"][1]["role"] == "user":
        nama_file_saja = os.path.basename(path_sumber)
        teks_user_lama = chat["messages"][1]["content"]

        # Kita langsung replace satu blok penuh agar rapi
        teks_tag_lama = f'[Konteks: "{nama_file_saja}"]'
        teks_tag_baru = f'[Konteks:\n{teks_konteks.strip()}\n]'

        teks_user_baru = teks_user_lama.replace(teks_tag_lama, teks_tag_baru)

        chat["messages"][1]["content"] = teks_user_baru

    # ---------------------------------------------------------
    # HAPUS METADATA AGAR SIAP TRAINING
    # ---------------------------------------------------------
    if "metadata" in chat:
        del chat["metadata"]

os.makedirs(os.path.dirname(FILE_JSON_BARU), exist_ok=True)

# 4. Simpan hasil akhirnya menjadi file baru
with open(FILE_JSON_BARU, "w", encoding="utf-8") as f:
    json.dump(data_mentah, f, indent=4, ensure_ascii=False)

print("Proses Selesai! File baru berhasil disimpan dan siap ditraining.")

Ditemukan 1667 data.
Proses Selesai! File baru berhasil disimpan dan siap ditraining.
